# Streaming Pipeline — Inserción directa en `bronze.bronze_bookings`

Este notebook implementa la **ingesta event-driven** de nuevos registros
directamente en la tabla `bronze.bronze_bookings`, cumpliendo la sección 3.1
de la guía del proyecto.

## Arquitectura

```
Productor sintético (PySpark)
         │ genera eventos de reserva
         ▼
   INSERT INTO bronze.bronze_bookings
         │ (modo append)
         ▼
   bronze.bronze_bookings   ← TABLA ÚNICA (batch + nuevos eventos)
         │
         ▼
   dbt run --select silver    ← procesa todo, sin distinción de origen
         │
         ▼
   dbt run --select gold      ← Star Schema actualizado
         │
         ▼
   Power BI Dashboard refresca
```

## ¿Por qué insertar directo y no usar Structured Streaming?

**Decisión arquitectónica defendible**:

1. **Una sola tabla por entidad** — patrón estándar de marketplaces (Snowflake, Iceberg).
   Evita tablas intermedias `_stream`, `_topic`, `_all` que complican el linaje.
2. **Silver/Gold no necesitan saber el origen** — procesan la tabla Bronze unificada
   sin distinguir si un registro vino por batch inicial o por inserción event-driven.
3. **Power BI simplificado** — consume la tabla única vía DirectQuery; no necesita
   una vista que una múltiples fuentes.
4. **Demo en vivo más clara** — INSERT → Silver procesa → Gold se actualiza → Dashboard cambia.

**Migrar a Structured Streaming en el futuro** sería trivial: reemplazar el `INSERT INTO`
por un `readStream.format("kafka")` que escriba a la misma tabla. El resto del pipeline
(Silver, Gold, dashboards) no requiere cambios.

## 1. Configuración

In [ ]:
# Catálogo actual (auto-detectado)
CATALOG_NAME = spark.sql("SELECT current_catalog()").collect()[0][0]

# Número de eventos sintéticos a generar e insertar
N_EVENTOS = 50

# Tabla destino — la MISMA que carga el notebook 02 (Bronze inicial)
TARGET_TABLE = "bronze.bronze_bookings"

print(f"Catálogo:           {CATALOG_NAME}")
print(f"Eventos a generar: {N_EVENTOS}")
print(f"Tabla destino:      {TARGET_TABLE}")

## 2. Verificar estado actual de Bronze

Contamos los registros que ya existen para después comparar contra el estado
después de la inserción.

In [ ]:
%sql
SELECT COUNT(*) AS registros_antes
FROM bronze.bronze_bookings;

## 3. Capturar rangos válidos de IDs desde Silver

Los `user_id` y `property_id` de los eventos sintéticos deben caer dentro de los
rangos observados en Silver para que los joins en Gold no produzcan huérfanos.

In [ ]:
from pyspark.sql.functions import min as F_min, max as F_max

lim_u = spark.table("silver.silver_users").agg(
    F_min("user_id").alias("u_min"),
    F_max("user_id").alias("u_max")
).collect()[0]
USER_MIN, USER_MAX = lim_u["u_min"], lim_u["u_max"]

lim_p = spark.table("silver.silver_properties").agg(
    F_min("property_id").alias("p_min"),
    F_max("property_id").alias("p_max")
).collect()[0]
PROP_MIN, PROP_MAX = lim_p["p_min"], lim_p["p_max"]

# Obtener el último booking_id para generar IDs nuevos sin colisión
max_id = spark.table(TARGET_TABLE).agg(F_max("booking_id").alias("m")).collect()[0]["m"]
NEXT_BOOKING_ID = (max_id or 0) + 1

print(f"user_id     range: [{USER_MIN}, {USER_MAX}]")
print(f"property_id range: [{PROP_MIN}, {PROP_MAX}]")
print(f"Próximos booking_id desde: {NEXT_BOOKING_ID}")

## 4. Generar eventos sintéticos

Usa `spark.range(N_EVENTOS)` para crear el lote de eventos con datos coherentes
con el esquema de `bronze.bronze_bookings`.

In [ ]:
eventos_nuevos = spark.range(N_EVENTOS).selectExpr(
    f"({NEXT_BOOKING_ID} + id) AS booking_id",
    f"CAST({USER_MIN} + rand() * ({USER_MAX} - {USER_MIN}) AS BIGINT) AS user_id",
    f"CAST({PROP_MIN} + rand() * ({PROP_MAX} - {PROP_MIN}) AS BIGINT) AS property_id",
    "date_add(current_date(), CAST(rand() * 90 AS INT)) AS check_in",
    "date_add(current_date(), CAST(rand() * 90 AS INT) + CAST(1 + rand() * 13 AS INT)) AS check_out",
    "CAST(1 + rand() * 5 AS INT) AS guests_count",
    "ROUND(50 + rand() * 1450, 2) AS total_amount",
    "CASE WHEN rand() < 0.5 THEN 'confirmed' WHEN rand() < 0.8 THEN 'pending' ELSE 'cancelled' END AS status",
    "current_timestamp() AS created_at",
    "current_timestamp() AS updated_at"
)

print("Esquema de los eventos generados:")
eventos_nuevos.printSchema()

print(f"\nMuestra de 5 eventos:")
eventos_nuevos.show(5, truncate=False)

## 5. Insertar los eventos directamente en `bronze.bronze_bookings`

Usamos `mode("append")` para agregar los nuevos registros sin sobrescribir los
existentes. Es la operación equivalente a `INSERT INTO ... SELECT ...` en SQL.

In [ ]:
(
    eventos_nuevos.write
                  .format("delta")
                  .mode("append")
                  .saveAsTable(TARGET_TABLE)
)

print(f"✓ Insertados {N_EVENTOS} eventos nuevos en {TARGET_TABLE}")

## 6. Verificar el resultado

Contamos los registros después de la inserción y mostramos los más recientes.

In [ ]:
%sql
SELECT COUNT(*) AS registros_despues
FROM bronze.bronze_bookings;

In [ ]:
%sql
-- Ver los 10 eventos más recientes (los recién insertados)
SELECT booking_id, user_id, property_id, check_in, check_out, total_amount, status, created_at
FROM bronze.bronze_bookings
ORDER BY created_at DESC
LIMIT 10;

## 7. Próximo paso — Propagar los nuevos eventos a Silver y Gold

Para que los dashboards de Power BI vean estos nuevos eventos:

1. Ejecutar el notebook `08_run_dbt.ipynb` (o `dbt build` desde la línea de comandos).
   - Silver se reconstruye con los nuevos eventos en `silver.silver_bookings`.
   - Gold se reconstruye con la fact table actualizada (`gold.gold_fact_reservas`).
2. En Power BI, presionar **Actualizar**. Los KPIs (GMV, Reservas Totales) suben
   reflejando los nuevos eventos.

Este flujo demuestra el pipeline **end-to-end**: Bronze ← evento → Silver (dbt) → Gold (dbt) → Power BI.

## Conclusión

Este notebook simula la llegada de **N eventos sintéticos** directamente en
`bronze.bronze_bookings`, manteniendo una arquitectura limpia con **una sola
tabla por entidad**.

### Decisiones de diseño defendibles

- **¿Por qué insertar directo en `bronze_bookings` en vez de tener tablas intermedias?**
  Para evitar el anti-patrón de tener `bronze_bookings_stream`, `bronze_events_topic`
  y `bronze_bookings_all` que complican el linaje sin aportar valor. En arquitecturas
  Lakehouse modernas (Snowflake, Iceberg), una entidad = una tabla, sin distinción
  de origen a nivel físico.

- **¿Por qué no usamos Structured Streaming?** El proyecto demuestra el patrón
  productivo end-to-end. Migrar a Structured Streaming es reemplazar el INSERT
  por un `readStream.format("kafka")` que escriba a la misma tabla. El resto del
  pipeline (Silver, Gold, dashboards) no cambia. La arquitectura está documentada
  en `documentation/arquitectura.md` para soportar ambos modos.

- **¿Por qué `mode("append")` y no `MERGE INTO`?** Los nuevos eventos tienen
  `booking_id` generados por encima del máximo existente (`MAX(booking_id) + 1`),
  garantizando que no hay colisión de PKs. `append` es más eficiente que `MERGE`
  cuando se sabe que no hay duplicados.

- **¿Por qué `rand()` con rangos reales de Silver?** Para que los `user_id` y
  `property_id` generados existan en las dimensiones de Gold y los joins no
  produzcan registros huérfanos.

### Para la demo en vivo

1. **Mostrar el conteo inicial**: `SELECT COUNT(*) FROM bronze.bronze_bookings` → X.
2. **Run all del notebook 06** → inserta 50 eventos nuevos.
3. **Conteo después**: `SELECT COUNT(*) FROM bronze.bronze_bookings` → X + 50.
4. **Correr notebook 08 (dbt run)** → Silver y Gold se actualizan.
5. **Refrescar Power BI** → KPIs suben.